# Customer Intelligence Platform — Notebook 3: Churn Prediction Pipeline
**Portfolio Project | Notebook 4 of 5**

---

## Learning Objectives
1. Split before preprocessing — every leakage-safe pipeline in this project starts here
2. Build a single leak-free `Pipeline` combining a custom transformer, a `ColumnTransformer` (numeric impute+scale, categorical one-hot), and a classifier
3. Write a custom `BaseEstimator`/`TransformerMixin` transformer whose fit-time statistic is learned from **training data only**
4. Quantify whether the Notebook 1 cluster `segment` label actually earns its place as a model feature, via a controlled ablation
5. Tune hyperparameters across the whole pipeline with `GridSearchCV`, then persist the final fitted pipeline as a single deployable artefact

> **Senior engineer framing:** This is the notebook where the segmentation work (Notebook 1) and the supervised-learning work meet. The key engineering discipline here isn't any individual algorithm — it's making sure `segment` flows into the churn model as *evidence to be tested*, not as an assumed-good feature. If it doesn't measurably improve the model, it doesn't go into the production pipeline, no matter how satisfying Notebook 1's silhouette plots were.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid", palette="deep")
RNG_SEED = 42

DATA_PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

df = pd.read_csv(DATA_PROCESSED_DIR / "customers_with_segments.csv")
df["segment"] = df["segment"].astype(str)  # treat as categorical, not numeric
df.shape

---
## Part 1: Split First — Always

### TODO 1 — Train/test split before touching any preprocessing

**HINT:** `train_test_split(df, test_size=0.2, stratify=df["churned"], random_state=RNG_SEED)` — stratify on the target since `churned` is imbalanced (check the rate first). Everything fit in this notebook (imputer medians, scaler means/stds, one-hot categories, the custom transformer's threshold) must be fit on `train_df` only.

In [ ]:
print("Overall churn rate:", df["churned"].mean().round(3))

# TODO: split df into train_df/test_df, stratified on "churned"
train_df, test_df = ...

target_col = "churned"
drop_cols = ["customer_id", target_col]

X_train, y_train = train_df.drop(columns=drop_cols), train_df[target_col]
X_test, y_test = test_df.drop(columns=drop_cols), test_df[target_col]
print(X_train.shape, X_test.shape)

---
## Part 2: A Custom Transformer With a Fit-Time-Learned Statistic

We'll engineer a `long_inactive_flag` feature — 1 if a customer's `recency_days` is above the 75th percentile **of the training set**. This is a deliberately chosen example because the threshold *must* come from `X_train` alone — computing it from the full dataset (train+test combined) would leak test-set distributional information into a feature seen at train time.

### TODO 2 — Implement `RecencyRiskFlagger`

**HINT:**
- `fit(self, X, y=None)`: compute `self.threshold_ = np.percentile(X["recency_days"], self.percentile)`, then `return self`
- `transform(self, X)`: work on a **copy** of `X` (never mutate the input in place), add a column `long_inactive_flag = (X["recency_days"] > self.threshold_).astype(int)`, return the modified DataFrame
- Inherit from `BaseEstimator, TransformerMixin` — this gets you `.fit_transform()` for free and makes the class compatible with `Pipeline`/`GridSearchCV`

In [ ]:
class RecencyRiskFlagger(BaseEstimator, TransformerMixin):
    """Adds a binary flag for customers whose recency_days exceeds a
    training-set-derived percentile threshold."""

    def __init__(self, percentile: float = 75):
        self.percentile = percentile

    def fit(self, X: pd.DataFrame, y=None):
        # TODO: learn self.threshold_ from X["recency_days"] using np.percentile
        self.threshold_ = ...
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        # TODO: copy X, add the "long_inactive_flag" column, return it
        X = ...
        X["long_inactive_flag"] = ...
        return X


# Quick sanity check on the training fold alone
flagger_check = RecencyRiskFlagger().fit(X_train)
print("Learned threshold (days):", flagger_check.threshold_)
flagger_check.transform(X_train.head())[["recency_days", "long_inactive_flag"]]

---
## Part 3: `ColumnTransformer` — Numeric and Categorical Branches

### TODO 3 — Build the numeric and categorical sub-pipelines and combine them

**HINT:**
- `numeric_cols` = the 10 behavioral features **plus** `"long_inactive_flag"` (added by the custom transformer upstream)
- `categorical_cols` = `["contract_type", "region", "acquisition_channel", "payment_method", "segment"]`
- Numeric branch: `Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())])`
- Categorical branch: `Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])` — `handle_unknown="ignore"` matters here: a category seen at inference time that never appeared in training (e.g. a brand-new `acquisition_channel`) would otherwise crash the pipeline
- Combine with `ColumnTransformer([("num", numeric_pipe, numeric_cols), ("cat", categorical_pipe, categorical_cols)])`

In [ ]:
numeric_cols = [
    "tenure_months", "monthly_spend", "total_spend_lifetime", "recency_days",
    "frequency_12m", "support_tickets_12m", "discount_usage_rate",
    "avg_session_minutes", "num_products", "satisfaction_score", "long_inactive_flag",
]
categorical_cols = ["contract_type", "region", "acquisition_channel", "payment_method", "segment"]

# TODO: build numeric_pipe, categorical_pipe, and the combined ColumnTransformer
numeric_pipe = ...
categorical_pipe = ...
preprocessor = ...
preprocessor

---
## Part 4: Assemble and Cross-Validate the Full Pipeline

### TODO 4 — Compose `RecencyRiskFlagger` → `preprocessor` → classifier into one `Pipeline`, then cross-validate

**HINT:** `Pipeline([("risk_flag", RecencyRiskFlagger()), ("preprocessor", preprocessor), ("model", LogisticRegression(max_iter=1000, random_state=RNG_SEED))])`. Use `cross_val_score(pipeline, X_train, y_train, cv=StratifiedKFold(5, shuffle=True, random_state=RNG_SEED), scoring="roc_auc")` — the *whole pipeline* is refit on each fold, so the custom transformer's threshold and the scaler's statistics are recomputed per fold, never touching that fold's held-out rows.

In [ ]:
# TODO: build the full pipeline with LogisticRegression as a baseline
baseline_pipeline = ...

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)
# TODO: run cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring="roc_auc")
baseline_cv_scores = ...
print(f"Baseline LogisticRegression ROC-AUC: {baseline_cv_scores.mean():.3f} +/- {baseline_cv_scores.std():.3f}")

### TODO 5 — Compare against RandomForest and GradientBoosting

**HINT:** build a `dict` of `{name: pipeline}` for `LogisticRegression`, `RandomForestClassifier(n_estimators=300, random_state=RNG_SEED)`, and `GradientBoostingClassifier(random_state=RNG_SEED)`, loop over it running the same `cross_val_score` call, and collect results into a small comparison table.

In [ ]:
candidate_models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=RNG_SEED),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=RNG_SEED),
    "gradient_boosting": GradientBoostingClassifier(random_state=RNG_SEED),
}

results = []
for name, model in candidate_models.items():
    # TODO: build a Pipeline([("risk_flag", ...), ("preprocessor", ...), ("model", model)])
    pipe = ...
    scores = ...  # cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    results.append({"model": name, "roc_auc_mean": scores.mean(), "roc_auc_std": scores.std()})

results_df = pd.DataFrame(results).sort_values("roc_auc_mean", ascending=False)
results_df

---
## Part 5: Ablation — Does the Cluster `segment` Feature Actually Help?

Notebook 1 produced a segmentation with a respectable silhouette score, but that says nothing about whether `segment` improves *churn prediction specifically*. Test it directly, with everything else held constant.

### TODO 6 — Cross-validate the best model from above with and without `segment` in `categorical_cols`

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = candidate_models[best_model_name]

categorical_cols_no_segment = [c for c in categorical_cols if c != "segment"]

# TODO: build a second ColumnTransformer using categorical_cols_no_segment instead of categorical_cols,
# wrap it in a Pipeline the same way as before, and cross-validate with cv/scoring="roc_auc"
preprocessor_no_segment = ...
pipeline_no_segment = ...
scores_no_segment = ...

print(f"With segment feature:    {results_df.iloc[0]['roc_auc_mean']:.4f}")
print(f"Without segment feature: {scores_no_segment.mean():.4f}")

**Interpretation:** if the two scores are within roughly one standard deviation of each other, `segment` is not earning its place as a churn-model feature — despite being a perfectly valid, well-validated segmentation for *marketing* purposes. A segmentation can be simultaneously "good" (stable, well-separated) and "not useful for this particular downstream model" — these are different questions, and conflating them is a common mistake.

---
## Part 6: Hyperparameter Tuning Across the Whole Pipeline

### TODO 7 — `GridSearchCV` over both preprocessing and model hyperparameters

**HINT:** use the winning architecture (with or without `segment`, based on your Part 5 finding) and tune with the double-underscore naming convention, e.g. `"preprocessor__num__impute__strategy": ["median", "mean"]` and, if the winner was `RandomForestClassifier`, `"model__max_depth": [4, 8, None]` / `"model__n_estimators": [200, 400]`.

In [ ]:
final_pipeline = Pipeline([
    ("risk_flag", RecencyRiskFlagger()),
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=RNG_SEED)),
])

# TODO: define a param_grid using the "step__param" and "step__substep__param" nesting convention
param_grid = {
    "risk_flag__percentile": [70, 75, 80],
    "preprocessor__num__impute__strategy": ["median", "mean"],
    "model__n_estimators": [200, 400],
    "model__max_depth": [4, 8, None],
}

# TODO: run GridSearchCV(final_pipeline, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
grid_search = ...
grid_search.fit(X_train, y_train)

print("Best CV ROC-AUC:", grid_search.best_score_)
print("Best params:", grid_search.best_params_)

---
## Part 7: Final Evaluation on the Held-Out Test Set

This is the **one and only** time `X_test`/`y_test` gets used. If you find yourself wanting to go back and adjust the pipeline based on this score, that's a sign the test set has stopped being a trustworthy final check.

In [ ]:
best_pipeline = grid_search.best_estimator_
y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["retained", "churned"]))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["retained", "churned"]).plot(ax=ax, cmap="Blues")
plt.title("Confusion Matrix — Held-Out Test Set")
plt.show()

### TODO 8 — Inspect feature importances by name

**HINT:** `best_pipeline.named_steps["preprocessor"].get_feature_names_out()` gives you the expanded feature names (including each one-hot category) in the same order as `best_pipeline.named_steps["model"].feature_importances_` — zip them together and sort descending.

In [ ]:
# TODO: pull out feature names and importances, build a sorted DataFrame, plot the top 15 as a horizontal bar chart
feature_names = ...   # best_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = ...     # best_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values(
    "importance", ascending=False
)
importance_df.head(15).plot(kind="barh", x="feature", y="importance", figsize=(8, 6), legend=False)
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importances")
plt.tight_layout()
plt.show()

---
## Part 8: Persist the Final Pipeline

Save the entire fitted `Pipeline` — custom transformer, `ColumnTransformer`, and model — as a **single artefact**. Anything that consumes this file (a FastAPI endpoint, a CLI batch job) needs to load exactly one object and call `.predict()`/`.predict_proba()` on a raw, unprocessed DataFrame with the same input schema as `X_train`.

In [ ]:
joblib.dump(best_pipeline, MODELS_DIR / "churn_pipeline.joblib")

# Sanity check: reload and confirm predictions match
reloaded_pipeline = joblib.load(MODELS_DIR / "churn_pipeline.joblib")
assert np.allclose(reloaded_pipeline.predict_proba(X_test)[:, 1], y_proba), "Reloaded pipeline predictions diverged!"
print("Saved and verified models/churn_pipeline.joblib")

---
## Senior Engineer Notes & Best Practices

1. **Split before you fit anything, including a custom transformer's internal statistic.** `RecencyRiskFlagger`'s `threshold_` is exactly the kind of easy-to-miss leakage vector — it's not a `sklearn` built-in, so it's on you to make sure it only ever sees `X_train` during `.fit()`.
2. **`handle_unknown="ignore"` on `OneHotEncoder` is not optional in production** — a training set can never contain every category a live system will eventually see, and the default (`error`) will crash a service the first time it doesn't.
3. **An ablation beats an assumption.** "We already built a segmentation, so obviously the churn model should use it" is exactly the kind of claim that a two-line `cross_val_score` comparison can validate or kill in minutes.
4. **Test-set discipline is a one-way door.** Evaluate on `X_test`/`y_test` exactly once, at the end. If you're tempted to loop back and adjust hyperparameters after seeing the test score, you've turned your test set into a second validation set without realizing it.
5. **`get_feature_names_out()` is what makes a `ColumnTransformer`-based pipeline explainable after the fact** — without it, "feature 17 was important" is meaningless to anyone, including you in six months.
6. **The saved pipeline is the deployment unit, not the model class by itself.** `joblib.load("churn_pipeline.joblib").predict(raw_dataframe)` must work end-to-end from raw, unprocessed input — that's the entire point of building this as one `Pipeline` instead of a model plus a pile of separate preprocessing scripts.

## Key Takeaways
- Split → custom transformer → `ColumnTransformer` → model, all inside one `Pipeline`, refit per cross-validation fold — this is the only way to guarantee zero leakage end-to-end
- A custom transformer's fit-time statistics are exactly as leak-prone as a `StandardScaler`'s — treat them with the same discipline
- Cluster-derived features must earn their place in a supervised model via a real ablation, not be assumed useful because the clustering itself was well-validated
- The single serialized `Pipeline` artefact (`churn_pipeline.joblib`) is what Notebook 4's MLflow tracking will log, and what a future FastAPI/CLI serving layer would load directly